# 01 - Data exploration
What's in SciFact (the evidence corpus and claims) and in the logical-fallacy dataset. Turn on a GPU runtime only for notebook 04; this one runs fine on CPU.

In [ ]:
import os, sys, subprocess
if "google.colab" in sys.modules:  # Colab: clone the repo and install dependencies
    if not os.path.isdir("fact-and-logic"):
        subprocess.run(["git", "clone", "https://github.com/Anishkumar25/fact-and-logic.git"], check=True)
    os.chdir("fact-and-logic")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-train.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

In [ ]:
import pandas as pd
from src import data

corpus = data.corpus()
train, val = data.claims('train'), data.claims('validation')
print(corpus.shape, train.shape, val.shape)
corpus.head(3)

## Label balance
NOT_ENOUGH_INFO is a big slice of the validation set, so accuracy alone is misleading. That's why the project reports macro-F1.

In [ ]:
pd.concat({'train': train.label.value_counts(), 'validation': val.label.value_counts()}, axis=1)

## Abstract length and evidence size

In [ ]:
import matplotlib.pyplot as plt
corpus.sentences.map(len).plot.hist(bins=30, title='Sentences per abstract')
plt.show()
train[train.gold_doc.notna()].gold_sents.map(len).value_counts().sort_index().rename('claims').to_frame()

## Fallacy dataset
Check the column names here first. `src/fallacy.py` expects `source_article` (text) and `logical_fallacies` (label).

In [ ]:
from collections import Counter
from datasets import load_dataset

fd = load_dataset('tasksource/logical-fallacy')
print(fd)
print(fd['train'].column_names)
Counter(fd['train']['logical_fallacies']).most_common()